# Chapter 10 — RAG Generation & Evaluation

*Where we are:* retrieval feeds **grounded generation**, and we learn to *measure* the answer.

```
ranked chunks →[ context builder → citation-aware prompt → LLM ]→ grounded answer → eval
```

In [1]:
# === Chapter 10 · standard bootstrap (identical pattern in every notebook) ===
# Runs standalone on a fresh Google Colab VM *or* a local checkout.
import os, sys, subprocess

# After you push this repo to GitHub, put its URL here (one edit works for every chapter):
REPO_URL = "https://github.com/rsalehin/patent-rag-masterclass"   # e.g. "https://github.com/<you>/patent-rag-masterclass"
NEED_OCR = False
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    target = "/content/patent-rag-masterclass"
    if not os.path.isdir(target):
        if REPO_URL:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, target], check=True)
        else:
            raise RuntimeError("Set REPO_URL to this repo's GitHub URL (see README.md).")
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if NEED_OCR:
        subprocess.run(["apt-get", "install", "-y", "-q", "tesseract-ocr"], check=False)

# Ensure the repo root (containing patentrag/) is importable.
for _cand in [os.getcwd()] + [os.path.dirname(os.getcwd())]:
    if os.path.isdir(os.path.join(_cand, "patentrag")):
        if _cand not in sys.path:
            sys.path.insert(0, _cand)
        break

from patentrag import bootstrap as bs
bs.setup_environment(REPO_URL, need_ocr=NEED_OCR)
bs.set_seeds()
_env = bs.environment_report()
print("Chapter 10 bootstrap OK")
print("  Python", _env["python"], "| Colab:", _env["in_colab"], "| CPU cores:", _env["cpu_count"])
print("  torch", _env["torch"], "| CUDA:", _env["cuda_available"], "| tesseract:", _env["tesseract"])

Chapter 10 bootstrap OK
  Python 3.12.10 | Colab: False | CPU cores: 24
  torch 2.12.0.dev20260304+cu130 | CUDA: True | tesseract: True


## 43. Retrieval ≠ context construction

Good context is **deduplicated**, **source-diverse**, **ordered**, and **within budget** — not
just "top-k glued together". `ContextBuilder` enforces all four.

In [2]:
import pandas as pd
from patentrag.dense import DenseRetriever
from patentrag.fusion import reciprocal_rank_fusion
from patentrag.generation import ContextBuilder
chunks = bs.ensure("chunks"); by_id = {c.chunk_id: c for c in chunks}
bm25 = bs.ensure("bm25_index"); emb = bs.ensure("embeddings")
dense = DenseRetriever(emb["chunk_ids"], emb["matrix"])

query = "how are retrieval-aware embeddings used to search media objects?"
fused = reciprocal_rank_fusion([[c for c,_ in bm25.search(query,40)], [c for c,_ in dense.search(query,40)]])
ranked = [by_id[cid] for cid,_ in fused[:20]]
passages = ContextBuilder(token_budget=1000, max_per_doc=2).build(ranked)
print(f"from {len(ranked)} ranked chunks → {len(passages)} context passages "
      f"(≤2 per doc, ≤1000 tokens, deduped)")
pd.DataFrame([{"citation": p.citation.label()[:52], "tokens": p.tokens, "text": p.text[:40]} for p in passages])

C:\Users\rsalehin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9996.84it/s]

from 20 ranked chunks → 2 context passages (≤2 per doc, ≤1000 tokens, deduped)


,citation,tokens,text
0,[PATENT=US11971885B2 | SECTION=Image Search Ap...,99,The retrieval network 110 can efficientl
1,[PATENT=US11971885B2 | SECTION=Image Search Ap...,50,The database 120 stores the media object


## 44. Citation-aware prompt construction

Each passage carries a **stable citation tag** `[PATENT=… | SECTION/CLAIM=… | CHUNK=…]`, and the
system prompt requires the model to answer *with* those tags — making the output mechanically
verifiable downstream.

In [3]:
from patentrag.generation import build_prompt, SYSTEM_PROMPT
prompt = build_prompt(query, passages)
print("SYSTEM:", SYSTEM_PROMPT[:130], "...\n")
print("PROMPT (head):\n", prompt[:420], "...")

SYSTEM: You are a patent analysis assistant. Answer the question using ONLY the context passages. Every statement must cite the passage it ...

PROMPT (head):
 Context:

[PATENT=US11971885B2 | SECTION=Image Search Application | CHUNK=6acd8fe0d9a8]
The retrieval network 110 can efficiently and accurately compare a query object and a set of media objects in a common and sparse embedding space, and scale to millions of contents. As a result, the system can be further applied to multiple embeddings. For example, the query may include multiple embeddings such as a content embedd ...


## 45. Grounded generation — `LLMProvider` (no paid key required)

The default `MockLLMProvider` is **deterministic and extractive**: it answers only from the
supplied passages, echoing their tags — grounded *by construction*, which lets us exercise the
rest of the pipeline offline. An OpenAI-compatible endpoint is an **optional** env-gated path
(`LLM_BASE_URL`/`LLM_API_KEY`/`LLM_MODEL`).

In [4]:
import os
from patentrag.generation import generate_answer, MockLLMProvider, OpenAICompatibleProvider
live = OpenAICompatibleProvider.available()
print("live LLM configured:", live, "→ using", "live endpoint" if live else "deterministic mock")
answer = generate_answer(query, ranked, provider=(OpenAICompatibleProvider() if live else MockLLMProvider()))
print("\nANSWER:\n", answer.answer[:360])
print("\ncitations:", [c.label() for c in answer.citations])
# grounding invariant: every cited chunk was actually retrieved
retrieved_ids = {c.chunk_id for c in ranked}
print("all citations came from retrieved context:", all(c.chunk_id in retrieved_ids for c in answer.citations))

live LLM configured: False → using deterministic mock

ANSWER:
 The retrieval network 110 can efficiently and accurately compare a query object and a set of media objects in a common and sparse embedding space, and scale to millions of contents. [PATENT=US11971885B2 | SECTION=Image Search Application | CHUNK=6acd8fe0d9a8] The database 120 stores the media objects and the respective sparse embeddings for each of the media

citations: ['[PATENT=US11971885B2 | SECTION=Image Search Application | CHUNK=6acd8fe0d9a8]', '[PATENT=US11971885B2 | SECTION=Image Search Application | CHUNK=09c4ab951e4f]']
all citations came from retrieved context: True


## 46. Why LLM evaluation is hard

Generation is **nondeterministic**, answers can be **correct in many wordings**, and there is
often **no single reference**. LLM-as-judge helps but has biases: **position bias** (favouring
the first option), **verbosity bias** (favouring longer answers), and **self-preference**
(favouring its own family's style). Never treat a judge as ground truth — calibrate it against
humans, average repeated judgements, and pair it with deterministic checks.

## 47. Deterministic evaluation

Cheap, exact, reproducible checks first: schema validity, citation validity, source-id validity.

In [5]:
from patentrag.guardrails import verify_citations, validate_structured_output
checks = verify_citations(answer, retrieved_ids, by_id)
print("citation validity:")
for ch in checks:
    print(f"  {ch.citation_label[:46]:46} exists={ch.source_exists} retrieved={ch.was_retrieved} valid={ch.valid}")
ok, obj, err = validate_structured_output(answer.model_dump())
print("\nstructured-output schema valid:", ok)

citation validity:
  [PATENT=US11971885B2 | SECTION=Image Search Ap exists=True retrieved=True valid=True
  [PATENT=US11971885B2 | SECTION=Image Search Ap exists=True retrieved=True valid=True

structured-output schema valid: True


## 48. Semantic evaluation

**Answer relevance** (is the answer on-topic for the query?) and **faithfulness** (is every
answer sentence supported by the retrieved context?), both via embeddings.

In [6]:
from patentrag.evaluation import answer_relevance, faithfulness
ctx_texts = [by_id[c.chunk_id].text for c in answer.citations]
ar = answer_relevance(query, answer.answer_without_tags)
fa = faithfulness(answer.answer_without_tags, ctx_texts)
print(f"answer relevance (query ↔ answer): {ar:.2f}")
print(f"faithfulness (answer ⊑ context)  : {fa:.2f}")

answer relevance (query ↔ answer): 0.68
faithfulness (answer ⊑ context)  : 1.00


## 49. LLM-as-judge (rubric)

A judge scores an answer against a **rubric** (relevance, groundedness, citation quality, 1–5),
ideally with a reference answer, temperature 0, and repeated/majority voting. We define the
rubric and — with no live judge — compute a **deterministic proxy** from the semantic + citation
signals above (clearly labelled as a proxy, not a real LLM judgement).

In [7]:
rubric = {
    "relevance": "Does the answer address the query? (1-5)",
    "groundedness": "Is every claim supported by cited context? (1-5)",
    "citation_quality": "Are citations present, valid, and sufficient? (1-5)",
}
valid_frac = sum(c.valid for c in checks) / max(1, len(checks))
proxy_score = round(1 + 4 * (0.4*ar + 0.4*fa + 0.2*valid_frac), 2)   # map [0,1]→[1,5]
print("Rubric dimensions:", list(rubric))
print(f"deterministic proxy judge score (NOT a live LLM): {proxy_score}/5")
print("With LLM_BASE_URL/LLM_API_KEY set, swap in a real judge over this same rubric.")

Rubric dimensions: ['relevance', 'groundedness', 'citation_quality']
deterministic proxy judge score (NOT a live LLM): 4.49/5
With LLM_BASE_URL/LLM_API_KEY set, swap in a real judge over this same rubric.


## 50–52. Decompose RAG quality; core metrics; frameworks

RAG quality decomposes: **retriever** → **context** → **generator** → **answer**. A wrong answer
does *not* automatically implicate the LLM. Core metrics, computed manually over a benchmark
query:

- **context precision / recall** — of retrieved docs, how many are relevant / of relevant, how many retrieved
- **faithfulness** — answer supported by context
- **answer relevance** — answer on-topic for the query

In [8]:
from patentrag.evaluation import context_precision, context_recall, ranked_docs_from_chunks
ex = next(e for e in bs.ensure("eval_dataset") if "retrieval-aware" in e.query or "media objects" in e.query)
r_docs = ranked_docs_from_chunks([c for c,_ in fused[:10]], by_id)
cp = context_precision(r_docs[:5], ex.relevant_doc_ids)
cr = context_recall(r_docs[:5], ex.relevant_doc_ids)
ans2 = generate_answer(ex.query, ranked, provider=MockLLMProvider())
pd.DataFrame([{
    "context_precision@5": round(cp, 2), "context_recall@5": round(cr, 2),
    "faithfulness": round(faithfulness(ans2.answer_without_tags, [by_id[c.chunk_id].text for c in ans2.citations]), 2),
    "answer_relevance": round(answer_relevance(ex.query, ans2.answer_without_tags), 2),
}])

,context_precision@5,context_recall@5,faithfulness,answer_relevance
0,1.0,1.0,1.0,0.51


In [9]:
# Framework path (Ragas / DeepEval): described, run only if a live LLM judge is configured.
if OpenAICompatibleProvider.available():
    print("A live endpoint is configured — a Ragas/DeepEval run could be wired here.")
else:
    print("SKIPPED (loudly): Ragas/DeepEval faithfulness/answer-relevance metrics use an LLM judge")
    print("+ embeddings; with no LLM_API_KEY we implement the SAME metrics manually above so the")
    print("chapter stays fully executable offline. Set LLM_BASE_URL/LLM_API_KEY to enable the")
    print("framework path. (Ragas and DeepEval are the two current mainstream choices.)")

SKIPPED (loudly): Ragas/DeepEval faithfulness/answer-relevance metrics use an LLM judge
+ embeddings; with no LLM_API_KEY we implement the SAME metrics manually above so the
chapter stays fully executable offline. Set LLM_BASE_URL/LLM_API_KEY to enable the
framework path. (Ragas and DeepEval are the two current mainstream choices.)


## 53. RAG failure attribution (A–E)

Different failures need different metrics to detect. We stage five *illustrative* scenarios and
show which signal flags each.

In [10]:
gold_ctx = "Retrieval-aware embedding trains dense embeddings for the retrieval objective."
scenarios = [
    ("A: retriever failed", context_recall([], {"US11971885B2"}), "context_recall ↓"),
    ("B: right evidence buried by reranker", context_recall(["Xdoc", "US11971885B2"], {"US11971885B2"}), "context present but low rank → MRR ↓"),
    ("C: good context, LLM hallucinated", round(faithfulness("The system uses quantum entanglement.", [gold_ctx]), 2), "faithfulness ↓"),
    ("D: correct answer, wrong citation", "citation.valid = False", "citation verification ↓"),
    ("E: grounded but incomplete", round(faithfulness("Embeddings are dense.", [gold_ctx]), 2), "faithful but low answer_relevance/recall"),
]
pd.DataFrame([{"failure": s, "signal_value": v, "detected_by": d} for s, v, d in scenarios])

,failure,signal_value,detected_by
0,A: retriever failed,0.0,context_recall ↓
1,B: right evidence buried by reranker,1.0,context present but low rank → MRR ↓
2,"C: good context, LLM hallucinated",0.0,faithfulness ↓
3,"D: correct answer, wrong citation",citation.valid = False,citation verification ↓
4,E: grounded but incomplete,0.0,faithful but low answer_relevance/recall


**Production implications.** Log the decomposed metrics per request; an alert on *faithfulness*
means fix generation/prompting, while an alert on *context recall* means fix retrieval. Keep a
deterministic layer (citation + schema validity) that can hard-fail a response regardless of what
a judge says.

## Chapter invariants

In [11]:
assert answer.citations and all(c.chunk_id in retrieved_ids for c in answer.citations)
assert all(c.source_exists and c.was_retrieved for c in checks)
assert 0.0 <= ar <= 1.0 and 0.0 <= fa <= 1.0
assert ok  # structured output validates
assert context_recall(["US11971885B2"], {"US11971885B2"}) == 1.0
bs.save_artifact("rag_runs", {"query": query, "answer": answer.answer, "citations": len(answer.citations),
                              "answer_relevance": ar, "faithfulness": fa})
print("All Chapter 10 invariants hold. rag_runs artifact ready.")

All Chapter 10 invariants hold. rag_runs artifact ready.


In [12]:
# === Chapter 10 validation footer ===
import time, platform, sys, importlib.metadata as _md
_pkgs = ['sentence-transformers', 'pydantic', 'pandas']
print("Chapter 10 — environment")
print("  Python :", sys.version.split()[0], "on", platform.system(), platform.release())
for _p in _pkgs:
    try: print(f"  {_p:24}: {_md.version(_p)}")
    except Exception: print(f"  {_p:24}: (not installed)")
print()
print("CHAPTER 10 VALIDATION: PASS")

Chapter 10 — environment
  Python : 3.12.10 on Windows 11
  sentence-transformers   : 6.0.0
  pydantic                : 2.13.3
  pandas                  : 3.0.2

CHAPTER 10 VALIDATION: PASS
